In [7]:
!pip install -q yfinance

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from models.wmt_model import WMTTradingModel
from models.nvda_model import NVDATradingModel
from models.mpc_model import MPCTradingModel
from models.xom_model import XOMTradingModel
from models.VAR_model import VARTradingModel

from utils import ForecastingMetrics, TradingMetrics, PortfolioEvaluator
from backtest import Backtest
import yfinance as yf
import numpy as np
import pandas as pd


from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import plotting
import matplotlib.pyplot as plt
import pandas as pd
from pypfopt.expected_returns import mean_historical_return
from pypfopt.risk_models import CovarianceShrinkage

The best models for each stock are:
- WMT: VAR(0)
    - Provided lower error, higher robustness, and simpler logic
- MPC: ARIMA/ARIMA-GARCH
    - Captured both trend and shock-adjustment behavior
    - Although the MSE is slightly lower than the Naive Baseline, it has higher directional accuracy.
- NVDA: VAR(0)
    - This stock is very volatile 
    - More event-driven rather than dependent on historical data so 
        - enforcing lag-based structure increased forecast error 
    - The mean forecast (VAR(0)) minimized cumulative prediction error better than models trying to impose structure where none existed
- XOM: VAR(1)
    - Provided the best balance between low forecast error, directional accuracy, and alignment

In [3]:
# ChatGPT made this dictionary
stock_categories = {
    "AAPL": "Technology", "MSFT": "Technology", "NVDA": "Technology",
    "AMZN": "Technology", "GOOGL": "Technology", "META": "Technology", "TSLA": "Technology",
    "JPM": "Finance", "BAC": "Finance", "WFC": "Finance", "C": "Finance", "GS": "Finance", "MS": "Finance",
    "KO": "Consumer Goods", "PG": "Consumer Goods", "PEP": "Consumer Goods",
    "WMT": "Consumer Goods", "COST": "Consumer Goods",
    "CL": "Consumer Goods", "XOM": "Energy", "CVX": "Energy", "COP": "Energy",
    "SLB": "Energy", "EOG": "Energy", "MPC": "Energy",
    "SPY": "ETF", "QQQ": "ETF", "DIA": "ETF", "IWM": "ETF", "VTI": "ETF"
}

# Get the inverse of stock_categories
category_stock = {}
for stock, category in stock_categories.items():
    if category not in category_stock:
        category_stock[category] = []
    category_stock[category].append(stock)

In [8]:
# Define date range
start = "2022-01-01"
end = "2025-01-01"

# Download historical data
close_historical_df = yf.download(list(stock_categories.keys()), start=start, end=end, auto_adjust=True)["Close"]
close_historical_df.head()

[*********************100%***********************]  30 of 30 completed


Ticker,AAPL,AMZN,BAC,C,CL,COP,COST,CVX,DIA,EOG,...,PEP,PG,QQQ,SLB,SPY,TSLA,VTI,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2022-01-03,178.270340,170.404495,41.931015,54.393021,76.912796,63.700024,540.298584,101.482414,341.154022,75.843277,...,154.084122,147.359772,392.184082,29.220692,453.210327,399.926666,230.069717,45.828423,45.858704,55.125225
2022-01-04,176.007767,167.522003,43.574482,54.815403,77.167397,66.463219,537.934021,103.328979,343.197205,79.329262,...,154.306808,147.875412,387.097321,30.639347,453.058594,383.196655,229.634155,47.653259,45.018566,57.198723
2022-01-05,171.326004,164.356995,42.839016,54.177521,77.485596,65.323418,524.291077,104.001190,339.670898,77.873314,...,154.832367,148.544800,375.205292,30.639347,444.358917,362.706665,224.653412,47.237701,45.627266,57.910122
2022-01-06,168.465988,163.253998,43.701607,55.953262,76.976433,67.775734,524.176636,104.886177,338.084747,79.470718,...,154.867981,147.296432,374.941589,31.367104,443.941467,354.899994,224.596619,48.448235,45.500454,59.272209
2022-01-07,168.632477,162.554001,44.654991,56.703209,76.776405,69.632263,511.191254,106.392319,338.010162,81.667145,...,155.063950,147.215057,370.879974,32.269886,442.186340,342.320007,223.573929,49.478081,45.934788,59.758030


In [9]:
n = close_historical_df.shape[0]

# Compute split points
train_end = int(n * 0.7)           # 70% for training
val_end = train_end + int(n * 0.2) # next 20% for validation

In [21]:
# Get the log 
close_log_df = np.log(close_historical_df)
all_non_test_df = close_log_df.iloc[:val_end - 1]

# we minus one because we will difference
all_test_df = close_log_df.iloc[val_end - 2:]


In [22]:
all_test_df

Ticker,AAPL,AMZN,BAC,C,CL,COP,COST,CVX,DIA,EOG,...,PEP,PG,QQQ,SLB,SPY,TSLA,VTI,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2024-09-11,5.400034,5.217758,3.640301,4.012106,4.635787,4.589573,6.796307,4.870920,5.994253,4.728669,...,5.131304,5.126656,6.143005,3.648452,6.302776,5.429916,5.593988,3.956729,4.357717,4.653704
2024-09-12,5.400528,5.231109,3.634644,4.012630,4.637764,4.592785,6.812425,4.880520,6.000828,4.731379,...,5.139632,5.124065,6.152774,3.654248,6.311164,5.437253,5.602267,3.915697,4.368064,4.667373
2024-09-13,5.399315,5.228378,3.631286,4.018543,4.632011,4.598599,6.813484,4.890526,6.008011,4.741060,...,5.140253,5.127576,6.157244,3.658260,6.316373,5.439339,5.609217,3.938889,4.379795,4.666653
2024-09-16,5.371147,5.219761,3.642862,4.030950,4.629169,4.618213,6.804569,4.900997,6.013874,4.760144,...,5.139407,5.145565,6.152816,3.669211,6.317849,5.423980,5.611199,3.957845,4.379299,4.680591
2024-09-17,5.373317,5.230467,3.654305,4.046246,4.620883,4.643841,6.792824,4.910244,6.013587,4.772560,...,5.137486,5.137863,6.153344,3.696074,6.318257,5.428775,5.612207,3.971141,4.354668,4.693549
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,5.549222,5.433940,3.775076,4.235314,4.502127,4.541836,6.860499,4.923387,6.056263,4.760387,...,4.999745,5.103467,6.269030,3.607156,6.390370,6.136171,5.684836,4.249458,4.521663,4.631162
2024-12-26,5.552393,5.425170,3.778899,4.240232,4.501911,4.539568,6.857700,4.924360,6.057904,4.756905,...,4.997321,5.110662,6.268351,3.607156,6.390436,6.118384,5.685408,4.251830,4.522849,4.632007
2024-12-27,5.539062,5.410529,3.774174,4.235314,4.495829,4.539877,6.840356,4.924499,6.050466,4.756822,...,5.000269,5.106953,6.254969,3.609010,6.379854,6.067638,5.674518,4.242731,4.510596,4.631913


In [40]:
target_stock = 'XOM'
related_stocks = category_stock[stock_categories[target_stock]]
xom_model = VARTradingModel(target_stock, related_stocks)

In [48]:
xom_model.fit(all_non_test_df[related_stocks])

1


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [35]:
all_train_df

Ticker,AAPL,AMZN,BAC,C,CL,COP,COST,CVX,DIA,EOG,...,PEP,PG,QQQ,SLB,SPY,TSLA,VTI,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2022-01-03,5.183301,5.138175,3.736026,3.996236,4.342672,4.154185,6.292122,4.619886,5.832334,4.328669,...,5.037499,4.992877,5.971731,3.374877,6.116357,5.991281,5.438383,3.824905,3.825565,4.009607
2022-01-04,5.170528,5.121115,3.774472,4.003971,4.345977,4.196649,6.287736,4.637918,5.838305,4.373607,...,5.038943,4.996370,5.958676,3.422285,6.116022,5.948548,5.436487,3.863951,3.807075,4.046532
2022-01-05,5.143568,5.102041,3.757449,3.992266,4.350092,4.179351,6.262047,4.644402,5.827977,4.355083,...,5.042343,5.000886,5.927473,3.422285,6.096633,5.893594,5.414559,3.855192,3.820505,4.058892
2022-01-06,5.126734,5.095307,3.777385,4.024517,4.343500,4.216204,6.261829,4.652876,5.823297,4.375388,...,5.042573,4.992447,5.926770,3.445760,6.095693,5.871836,5.414306,3.880496,3.817722,4.082140
2022-01-07,5.127722,5.091010,3.798966,4.037831,4.340897,4.243228,6.236744,4.667133,5.823076,4.402652,...,5.043838,4.991895,5.915878,3.474134,6.091731,5.835746,5.409742,3.901530,3.827223,4.090304
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-02-01,5.220960,5.070664,3.470195,3.962504,4.415992,4.651318,6.547901,4.908975,5.923880,4.668981,...,5.084778,5.025663,6.035025,3.843642,6.171302,5.241006,5.470830,3.845709,4.007790,4.559203
2024-02-02,5.215540,5.146389,3.467807,3.962684,4.398421,4.644562,6.554974,4.937965,5.927302,4.653317,...,5.080517,5.018792,6.051784,3.843642,6.181774,5.235963,5.479350,3.854494,4.015248,4.555092
2024-02-05,5.225339,5.137620,3.453362,3.943781,4.392138,4.638578,6.557339,4.938556,5.919925,4.642433,...,5.080224,5.019551,6.050478,3.834416,6.178126,5.198828,5.474608,3.845093,4.009868,4.550965


In [27]:
# Get the log 
close_log_df = np.log(close_historical_df)
xom_model.fit(close_log_df[related_stocks])

0


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
# Get the tickers
tickers = ["WMT", "NVDA", "MPC", "XOM"]
model_map = {
    "WMT": VARTradingModel,          # VAR(0) 
    "NVDA": VARTradingModel,        # VAR(0)
    "MPC": MPCTradingModel,          # ARIMA(2,1,1)/ARIMA-GARCH
    "XOM": VARTradingModel,          # VAR(1)
}

# for each ticker, get the adjusted closing price of the stock and the related stocks

# transform to be the log price

# fit the var model or arima model

# get the forecasted log price

In [9]:
np.log(closes)

Ticker,COP,CVX,EOG,MPC,SLB,XOM
Date,,,,,,
2022-01-03,4.154185,4.619886,4.328669,4.093457,3.374877,4.009608
2022-01-04,4.196649,4.637918,4.373607,4.125676,3.422285,4.046531
2022-01-05,4.179350,4.644402,4.355083,4.131998,3.422285,4.058892
2022-01-06,4.216204,4.652876,4.375388,4.159603,3.445760,4.082141
2022-01-07,4.243228,4.667133,4.402652,4.170660,3.474135,4.090304
...,...,...,...,...,...,...
2024-12-24,4.541836,4.923387,4.760387,4.885902,3.607156,4.631162
2024-12-26,4.539568,4.924360,4.756905,4.886198,3.607156,4.632007
2024-12-27,4.539877,4.924499,4.756822,4.888264,3.609010,4.631913


In [7]:
def forecast_and_markowitz(as_of_date: str,
                           horizon: int = 60,
                           lookback_days: int = 252,
                           objective: str = "gmir"
                           ) -> tuple[pd.DataFrame, pd.Series]:
    """
    For any date after Jan 2025, re-optimize the portfolio using
    forecasted returns from the best model per stock.

    Parameters
    ----------
    as_of_date : str
        Date string "YYYY-MM-DD". Use any date >= "2025-01-01".
        All data strictly before this date is used for training.
    horizon : int, default=60
        Number of future trading days to forecast.
    lookback_days : int, default=252
        Number of past trading days used for model estimation.
    objective : {"gmir", "gmv"}, default="gmir"
        Markowitz objective: max information ratio or min variance.

    Returns
    -------
    forecast_df : pd.DataFrame
        Shape (horizon, 4) with columns ["WMT", "NVDA", "MPC", "XOM"]
        containing forecasted daily returns.
    weights : pd.Series
        Markowitz optimal weights indexed by ticker, summing to 1.
    """
    tickers = ["WMT", "NVDA", "MPC", "XOM"]
    model_map = {
        "WMT": WMTTradingModel,          # VAR(0) 
        "NVDA": NVDATradingModel,        # VAR(0)
        "MPC": MPCTradingModel,          # ARIMA(2,1,1)/ARIMA-GARCH
        "XOM": XOMTradingModel,          # VAR(1)
    }

    end = pd.to_datetime(as_of_date)
    start = end - pd.tseries.offsets.BDay(int(lookback_days * 1.5))

    forecasts_list: list[np.ndarray] = []

    for t in tickers:
        # 1) download prices up to as_of_date
        data = yf.download(t, start=start, end=end)
        closes = data["Close"].dropna().tail(lookback_days)

        # 2) compute log price
        log_price = np.log(closes).values

        # 3) fit best model and produce 60-day forecast
        model_cls = model_map[t]
        model = model_cls()
        model.fit(log_price)

        dummy_X = np.zeros(horizon)
        fc = np.asarray(model.predict(dummy_X), dtype=float)
        forecasts_list.append(fc)

    # 4) stack forecasts of log prices into (horizon, n_assets) matrix
    log_price_forecast_matrix = np.column_stack(forecasts_list)
    log_price_forecast_df = pd.DataFrame(log_price_forecast_matrix, columns=tickers)

    price_forecast_df = np.exp(log_price_forecast_df)


    # # 5) compute Markowitz weights using forecasted returns
    # weights_arr, _ = _markowitz_from_forecasts(forecast_matrix, objective=objective)
    # weights = pd.Series(weights_arr, index=tickers, name="weight")

    return price_forecast_df, start, end#, weights

In [35]:
start

Timestamp('2023-08-23 00:00:00')

In [36]:
end

Timestamp('2025-02-01 00:00:00')

In [8]:
#forecast_df, weights = forecast_and_markowitz("2025-02-01")
forecast_df, start, end = forecast_and_markowitz("2025-02-01")

print("First 5 of the 60-day forecasts:")
display(forecast_df.head())

print("\nMarkowitz weights from forecasted returns:")
print(weights)

/var/folders/80/nt46s_zj37q09s9bvd_1j0vw0000gn/T/ipykernel_29464/2880841345.py:45: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(t, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/var/folders/80/nt46s_zj37q09s9bvd_1j0vw0000gn/T/ipykernel_29464/2880841345.py:45: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(t, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/var/folders/80/nt46s_zj37q09s9bvd_1j0vw0000gn/T/ipykernel_29464/2880841345.py:45: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(t, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressi

First 5 of the 60-day forecasts:


,WMT,NVDA,MPC,XOM
0,71.508766,111.544783,142.544835,103.105764
1,71.508766,111.544783,142.544835,103.162381
2,71.508766,111.544783,142.544835,103.217877
3,71.508766,111.544783,142.544835,103.272273
4,71.508766,111.544783,142.544835,103.325592



Markowitz weights from forecasted returns:


NameError: name 'weights' is not defined

In [18]:
mu = mean_historical_return(forecast_df, log_returns=True)
Sigma = CovarianceShrinkage(forecast_df, log_returns=True).ledoit_wolf()

# Compute tangency portfolio
ef_tan = EfficientFrontier(mu, Sigma)
tangency_portfolio = ef_tan.max_sharpe(risk_free_rate=0.02)

weights_df = pd.DataFrame(
    [(stock, weight) for stock, weight in ef_tan.clean_weights().items()],
    columns=['Stock', 'Weight']
)

weights_df

,Stock,Weight
0,WMT,0.0
1,NVDA,0.0
2,MPC,0.0
3,XOM,1.0
